# 02 · Modelling: baselines, gradient boosting, tuning, ablations
Every model is evaluated with **5-fold stratified CV on the training split** (AUC). The out-of-fold (OOF)
predictions are kept for calibration and threshold selection in notebook 03, so the holdout stays clean.

> Runtime: **GPU (T4)** recommended (XGBoost and CatBoost use it). Total time is about 60–90 min. Set `QUICK = True` for a ~15 min run.

In [ ]:
# ---------- Setup: run this cell first ----------
import os, sys, subprocess, json, warnings
warnings.filterwarnings("ignore")
SMOKE = os.environ.get("CR_SMOKE") == "1"          # local smoke test on synthetic data (ignore on Colab)
if SMOKE:
    PROJECT_DIR = os.environ["CR_PROJECT_DIR"]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/masters-projects/credit-risk"   # <- change if you put it elsewhere
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT_DIR}/requirements-colab.txt"], check=True)
sys.path.insert(0, f"{PROJECT_DIR}/src")
ART = f"{PROJECT_DIR}/artifacts"; FIG = f"{PROJECT_DIR}/reports/figures"
os.makedirs(ART, exist_ok=True); os.makedirs(FIG, exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
SEED = 42

def savefig(name):
    plt.tight_layout(); plt.savefig(f"{FIG}/{name}.png", dpi=160, bbox_inches="tight"); plt.show()

def save_json(obj, name):
    with open(f"{ART}/{name}.json", "w") as f: json.dump(obj, f, indent=2, default=float)
print("Project dir:", PROJECT_DIR)

In [ ]:
QUICK = SMOKE or False          # True = 3 folds, fewer Optuna trials (for a fast first pass)
N_FOLDS = 3 if QUICK else 5
N_TRIALS = 4 if SMOKE else (10 if QUICK else 25)

from creditrisk import models as M
feat = pd.read_parquet(f"{ART}/features.parquet")
FEATURES = json.load(open(f"{ART}/feature_list.json"))["features"]
tr, ho = feat[feat.SPLIT == "train"].reset_index(drop=True), feat[feat.SPLIT == "holdout"].reset_index(drop=True)
X, y, X_ho = tr[FEATURES], tr["TARGET"].values, ho[FEATURES]
print(X.shape, X_ho.shape, "GPU:", M.has_gpu())
results = {}

## 1. Baseline: regularised logistic regression (median-impute + scale + one-hot)

In [ ]:
results["logreg"] = M.cv_logistic(X, y, X_ho, n_folds=N_FOLDS, seed=SEED)
results["logreg"].summary(y)

## 2. Gradient-boosted trees: LightGBM, XGBoost, CatBoost (default-ish params)

In [ ]:
results["lgbm"] = M.cv_lightgbm(X, y, X_ho, n_folds=N_FOLDS, seed=SEED)
results["lgbm"].summary(y)

In [ ]:
results["xgb"] = M.cv_xgboost(X, y, X_ho, n_folds=N_FOLDS, seed=SEED)
results["xgb"].summary(y)

In [ ]:
results["cat"] = M.cv_catboost(X, y, X_ho, n_folds=N_FOLDS, seed=SEED)
results["cat"].summary(y)

## 3. Class imbalance: does re-weighting help?
A common approach is to up-weight the minority class (`scale_pos_weight`). We compare ranking quality (AUC)
**and** probability quality (Brier, ECE): re-weighting usually leaves AUC about the same but distorts the probabilities.

In [ ]:
from creditrisk.evaluate import classification_summary
spw = (y == 0).sum() / (y == 1).sum()
results["lgbm_weighted"] = M.cv_lightgbm(X, y, X_ho, {"scale_pos_weight": spw}, n_folds=N_FOLDS, seed=SEED,
                                         name="LightGBM (weighted)")
imb = pd.DataFrame({k: classification_summary(y, results[k].oof) for k in ["lgbm", "lgbm_weighted"]}).T
imb[["auc", "pr_auc", "brier", "ece"]]

## 4. Hyper-parameter tuning (Optuna TPE, 3-fold CV on a 50% subsample, proxy LR=0.08)

In [ ]:
study = M.tune_lightgbm(X, y, n_trials=N_TRIALS, timeout_minutes=40, sample_frac=0.5 if not SMOKE else 1.0, seed=SEED)
best_params = study.best_params
print("best proxy AUC:", round(study.best_value, 4)); best_params

In [ ]:
results["lgbm_tuned"] = M.cv_lightgbm(X, y, X_ho, {**best_params, "learning_rate": 0.02}, n_folds=N_FOLDS,
                                      seed=SEED, name="LightGBM (tuned)")
save_json(best_params, "best_params")
results["lgbm_tuned"].summary(y)

In [ ]:
tr_df = study.trials_dataframe()
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(tr_df["number"], tr_df["value"], "o", alpha=.5, label="trial"); ax.plot(tr_df["number"], tr_df["value"].cummax(), label="best so far")
ax.set_xlabel("trial"); ax.set_ylabel("CV AUC (proxy)"); ax.legend(); ax.set_title("Optuna search")
savefig("02_optuna")

## 5. Ablation: how much does each data source add?
We add one relational table at a time to the application-only model (same params, 3-fold CV, LR=0.05).

In [ ]:
prefix_groups = [("application only", None), ("+ bureau", "BURO_"), ("+ previous apps", "PREV_"),
                  ("+ POS cash", "POS_"), ("+ instalments", "INST_"), ("+ credit card", "CC_")]
derived = ["TOTAL_DEBT_INCOME_RATIO", "TOTAL_ANNUITY_INCOME_RATIO", "HAS_BUREAU", "HAS_PREV"]
aux_prefixes = [p for _, p in prefix_groups if p]
base_cols = [c for c in FEATURES if not any(c.startswith(p) for p in aux_prefixes) and c not in derived]
cols, rows = list(base_cols), []
for label, prefix in prefix_groups:
    if prefix:
        cols += [c for c in FEATURES if c.startswith(prefix)]
    r = M.cv_lightgbm(X[cols], y, X_ho[cols].iloc[:1], {**best_params, "learning_rate": 0.05}, n_folds=3, seed=SEED,
                      name=label, verbose=False)
    rows.append({"features": label, "n_features": len(cols), "cv_auc": np.mean(r.fold_auc)})
    print(rows[-1])
ablation = pd.DataFrame(rows); ablation["delta_auc"] = ablation["cv_auc"].diff()
ablation.to_csv(f"{ART}/ablation_data_sources.csv", index=False); ablation

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(ablation["features"], ablation["cv_auc"], marker="o"); ax.set_ylabel("CV AUC")
ax.set_title("Cumulative contribution of each data source"); ax.tick_params(axis="x", rotation=20)
savefig("02_ablation_sources")

## 6. Blending (rank-average; weights chosen on OOF predictions only)

In [ ]:
blend_members = ["lgbm_tuned", "xgb", "cat"]
w, blend_auc = M.blend_weights({k: results[k].oof for k in blend_members}, y)
print("weights:", w, " OOF AUC:", round(blend_auc, 4))
blend_oof = M.apply_blend({k: results[k].oof for k in blend_members}, w)
blend_ho = M.apply_blend({k: results[k].holdout for k in blend_members}, w)

## 7. Model comparison (cross-validated, training split only)

In [ ]:
from sklearn.metrics import roc_auc_score
summary = pd.DataFrame([results[k].summary(y) for k in results])
summary = pd.concat([summary, pd.DataFrame([{"model": "Blend (rank avg)", "oof_auc": roc_auc_score(y, blend_oof)}])])
summary = summary.sort_values("oof_auc", ascending=False).reset_index(drop=True)
summary.to_csv(f"{ART}/cv_model_comparison.csv", index=False)
summary.round(4)

In [ ]:
from sklearn.metrics import roc_curve
fig, ax = plt.subplots(figsize=(5.5, 5))
for k in ["logreg", "lgbm_tuned", "xgb", "cat"]:
    fpr, tpr, _ = roc_curve(y, results[k].oof); ax.plot(fpr, tpr, label=f"{results[k].name} ({roc_auc_score(y, results[k].oof):.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.set_title("OOF ROC curves"); ax.legend()
savefig("02_roc")

In [ ]:
imp = pd.DataFrame({"feature": FEATURES,
                    "gain": np.mean([b.feature_importance("gain") for b in results["lgbm_tuned"].models], axis=0)})
imp = imp.sort_values("gain", ascending=False).head(25)
fig, ax = plt.subplots(figsize=(7, 7)); ax.barh(imp.feature[::-1], imp.gain[::-1], color="#4C72B0")
ax.set_title("LightGBM (tuned): top-25 features by gain"); savefig("02_importance_gain")

## 8. Save predictions and models for notebook 03

In [ ]:
oof = pd.DataFrame({"SK_ID_CURR": tr.SK_ID_CURR, "TARGET": y, **{k: r.oof for k, r in results.items()}, "blend": blend_oof})
hop = pd.DataFrame({"SK_ID_CURR": ho.SK_ID_CURR, "TARGET": ho.TARGET.values, **{k: r.holdout for k, r in results.items()}, "blend": blend_ho})
oof.to_parquet(f"{ART}/oof_predictions.parquet", index=False); hop.to_parquet(f"{ART}/holdout_predictions.parquet", index=False)
for i, b in enumerate(results["lgbm_tuned"].models):
    b.save_model(f"{ART}/lgbm_tuned_fold{i}.txt")
save_json({"n_folds": N_FOLDS, "blend_weights": w, "final_model": "lgbm_tuned",
           "imbalance": imb[["auc", "brier", "ece"]].to_dict(orient="index"),
           "cv": summary.to_dict(orient="records")}, "cv_results")
print("saved.")

## Observations (fill in after running)
* Baseline → boosted trees: ...
* Re-weighting: ...
* Which data sources matter most and why: ...